# Modelado de Área de Incendios con KNN

## ¿Cómo funciona KNN?
- KNN es un algoritmo de aprendizaje supervisado que, para predecir el valor de una instancia, busca los *k* vecinos más cercanos en el espacio de características.
- La cercanía se mide mediante una métrica de distancia (euclídea, Manhattan, etc.).
- En regresión, la predicción es la media (o ponderación) de los valores de los vecinos.

In [ ]:
import pandas as pd
import numpy as np
import random
from sklearn.model_selection import train_test_split, GridSearchCV, RepeatedKFold
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error


## 1. Carga de Datos y Transformación del Area
Cargamos el dataset y se aplicamos **transformación logarítmica** a la variable objetivo (`area`) para reducir asimetrías y valores extremos.


In [ ]:
data = pd.read_csv('forestfires.csv')  # Asume forestfires.csv en el directorio
data['area_log'] = np.log1p(data['area'])  # Transformación: log(1 + area)
y = data['area_log']  # El area log-transformada
X = data.drop(['area', 'area_log'], axis=1)  # Resto de atributos
display(X.head())
display(y.head())

## 2. Selección(filtro) de variables y preprocesamiento y Pipeline 
- Usamos **RobustScaler**: Escala variables numéricas reduciendo el efecto de outliers.
- Creamos un **Pipeline** que aplicando el preprocesamiento y luego KNN de regersión.
- OneHot con `get_dummies()`: Codifica las variables categóricas transformandolas a numericas.
- Usamos `corrwith()` para obtener las variables que tienen mas correlacion  con el area. 

In [ ]:
# Creacion del pipeline con preprocesamiento RobustScaler() y modelo de regresion KNeighborsRegressor()
model = Pipeline([
    ("scaler", RobustScaler()),
    ("regressor", KNeighborsRegressor())
])
# Codificación one-hot de 'month' y 'day'
X = pd.get_dummies(X, columns=['month', 'day'], drop_first=True).astype(float)

correalation = X.corrwith(y)        # correlación de cada columna de X con y
correalation = correalation.abs()           # valor absoluto
correalation = correalation.sort_values(ascending=False)
# Nos quedamos con las 8 mejores variables (menos area_log)
best_variables = correalation.index[1:8].tolist()
print('Variables seleccionadas:', best_variables)

# Filtramos las variables de X, solo dejamos las mejoes
X = X[best_variables]
display(X.head())


## 3. Division de los datos de entrenamiento 
- Utilizamos `train_test_split()` para separar los datos en un 80% de `entrenamiento` y 20% para realizar los `test`


In [ ]:
# División train/test, 80% entrenamineto y 20% para el test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## 4. Validación Cruzada y Búsqueda de Hiperparámetros
- Se utiliza **RepeatedKFold** con 10 folds y 30 repeticiones para tener una validación robusta.
- Probamos con distintos valores de `n_neighbors(k)`, `weights` y `metric`.
- **GridSearchCV** con métrica `neg_mean_squared_error` porque asume que el MSE mas alto es el
     mejor, asi que usamos el negativo.


In [ ]:
# Validacion cruzada
cv = RepeatedKFold(n_splits=5, n_repeats=30, random_state=42)
model, cv

# Seleccion de hiperparámetros
param_grid = {
    'regressor__n_neighbors': range(2, 25),
    'regressor__weights': ['uniform', 'distance'],
    'regressor__metric': ['euclidean', 'manhattan']
}

# Búsqueda de hiperparámetros
grid_search = GridSearchCV(model, param_grid, cv=cv, scoring='neg_mean_squared_error', n_jobs=-1)
grid_search.fit(X_train, y_train)
print('Mejores parámetros:', grid_search.best_params_)

## 5. Evaluación Final
- Se evaluúa el mejor modelo sobre todo el dataset (predicciones y anti-transformación).
- Cálculo de **MSE**, **MAE** y **R²** en la escala original:
- **MSE** (Mean Squared Error): promedio de los cuadrados de los errores (hectareas²).
- **MAE** (Mean Absolute Error): promedio de los valores absolutos de los errores en hectareas.
- **R²**  (coeficiente de determinación): proporción de la varianza explicada por el modelo (adimensional, de 0 a 1).
- **Error absoluto** diferencia |predicción – valor real|, medido en hectáreas.
- **% de error** (error absoluto / valor real) × 100, un porcentaje que indica el tamaño del error relativo al valor real.


In [ ]:
# Elegimos el mejor modelo
best_model = grid_search.best_estimator_

y_pred = best_model.predict(X_test)
y_pred_result = np.expm1(y_pred)
y_true_result = np.expm1(y_test)

# Calculamos las metricas
mse = mean_squared_error(y_true_result, y_pred_result)
mae = mean_absolute_error(y_true_result, y_pred_result)
r2 = r2_score(y_true_result, y_pred_result)

print(f"\n=== Métricas sobre test ===")
print(f'- MSE: {mse:.2f}')
print(f'- MAE: {mae:.2f}')
print(f'- R²: {r2:.2f}')

## 6. Predicciones de Ejemplo
Cogemos 5 predicciones aleatorias del dataset y cálculamos la prediccion del aera quemada, el error absoluto y el error relativo de cada una.


In [ ]:
# Predicciones de ejemplo
print("\n=== Predicciones de Ejemplo ===")
random_indices = random.sample(range(len(X_test)), 5)

for i, idx in enumerate(random_indices):
    sample = X_test.iloc[idx:idx+1]
    real_value = y_true_result.iloc[idx]
    prediction = np.expm1(best_model.predict(sample))[0]
    error = abs(prediction - real_value)
    
    print(f"\nMuestra {i+1} (índice {idx}):")
    print(f"- Real: {real_value:.2f} hectáreas")
    print(f"- Predicción: {prediction:.2f} hectáreas")
    print(f"- Error absoluto: {error:.2f} hectáreas")
    print(f"- % Error: {(error/(real_value+1e-6))*100:.1f}%")